# Scraping Data Ulasan Aplikasi Gojek (Google Play Store)

**Proyek Analisis Sentimen — Dicoding**

Notebook ini **hanya berfokus pada pengambilan data mentah (raw)** ulasan aplikasi **Gojek**
(`com.gojek.app`) dari Google Play Store menggunakan pustaka
[`google-play-scraper`](https://pypi.org/project/google-play-scraper/).

Seluruh tahap lanjutan — pembersihan, penghapusan duplikat, preprocessing, pelabelan, dan
pelatihan model — dilakukan pada **`notebook.ipynb`** (notebook pelatihan model).

- **Sumber data:** Google Play Store (ulasan publik), bahasa Indonesia (`lang='id'`, `country='id'`).
- **Output:** `gojek_reviews_raw.csv` (data mentah hasil scraping).

> Catatan etika: data diambil melalui API publik `google-play-scraper`, tanpa kredensial/login,
> dan hanya berupa ulasan publik. Sesuai dengan pedoman Dicoding mengenai scraping yang beretika.

In [1]:
# Instalasi pustaka (jalankan jika belum ter-install)
# !pip install google-play-scraper pandas

import time
import pandas as pd
from google_play_scraper import reviews, Sort

print('Pustaka berhasil di-import.')

Pustaka berhasil di-import.


## 1. Konfigurasi & Fungsi Scraping

Ulasan diambil bertahap (paginasi) menggunakan `continuation_token`. Setiap panggilan mengambil
hingga 200 ulasan, lalu token dipakai untuk batch berikutnya sampai target terpenuhi atau data habis.

In [2]:
APP_ID = 'com.gojek.app'   # ID aplikasi Gojek di Google Play Store
TARGET = 15000            # target jumlah ulasan mentah
BATCH = 200               # jumlah ulasan per permintaan


def scrape_reviews(app_id, target, batch_size=200, sleep=0.4):
    """Mengambil ulasan secara bertahap memakai continuation_token (raw, tanpa pembersihan)."""
    collected = []
    token = None
    batch_no = 0
    while len(collected) < target:
        result, token = reviews(
            app_id,
            lang='id',
            country='id',
            sort=Sort.NEWEST,
            count=batch_size,
            continuation_token=token,
        )
        if not result:
            print('Tidak ada data tambahan, berhenti.')
            break
        collected.extend(result)
        batch_no += 1
        if batch_no % 5 == 0 or len(collected) >= target:
            print(f'Batch {batch_no:>3} | total terkumpul: {len(collected)}')
        if token is None:
            print('continuation_token habis, berhenti.')
            break
        time.sleep(sleep)  # jeda sopan agar tidak membebani server
    return collected

In [3]:
raw_reviews = scrape_reviews(APP_ID, TARGET, BATCH)
print(f'\nTotal ulasan mentah terkumpul: {len(raw_reviews)}')

Batch   5 | total terkumpul: 1000


Batch  10 | total terkumpul: 2000


Batch  15 | total terkumpul: 3000


Batch  20 | total terkumpul: 4000


Batch  25 | total terkumpul: 5000


...

Batch  60 | total terkumpul: 12000


Batch  65 | total terkumpul: 13000


Batch  70 | total terkumpul: 14000


Batch  75 | total terkumpul: 15000



Total ulasan mentah terkumpul: 15000


## 2. Simpan Data Mentah (Raw)

Kita hanya memilih kolom yang relevan, lalu menyimpan **apa adanya** (tanpa deduplikasi/pembersihan).
Tahap pembersihan & deduplikasi dilakukan di notebook pelatihan model.

In [4]:
df = pd.DataFrame(raw_reviews)[['reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'at']]
print('Ukuran data mentah:', df.shape)
df.head()

Ukuran data mentah: (15000, 6)


,reviewId,userName,content,score,thumbsUpCount,at
0,28bf2b5a-28c6-4e7b-abae-af2897e8c62d,Theem Theem Theem Theem,ramah ramah yang jemput,5,0,2026-05-23 14:27:43
1,e58ea6e2-793b-48e9-b8d3-c118d710d00c,Maksum Pontianak,jos gandos,5,0,2026-05-23 14:10:40
2,6fcacbb4-12af-417b-b084-46b68a546b71,Andi Anfal,trima kah pak,4,0,2026-05-23 14:02:21
3,0cb15383-c759-417a-9e00-3a8715526fa4,sancho gaming321,"posisi driver selalu ga gerak, seperti malas n...",1,0,2026-05-23 14:01:06
4,be2e83cf-ece4-43c7-b337-c90bb7cee4ef,masridha goiya,keren,5,0,2026-05-23 13:56:31


In [5]:
# Distribusi rating bintang (informasi awal saja)
print('Distribusi skor rating:')
print(df['score'].value_counts().sort_index())

Distribusi skor rating:
score
1    4222
2     753
3     622
4     699
5    8704
Name: count, dtype: int64


In [6]:
OUT_PATH = 'gojek_reviews_raw.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Dataset mentah disimpan ke: {OUT_PATH}')
print(f'Jumlah baris: {len(df)} | Jumlah kolom: {df.shape[1]}')

Dataset mentah disimpan ke: gojek_reviews_raw.csv
Jumlah baris: 15000 | Jumlah kolom: 6
